импорты

In [ ]:
!pip -q install beautifulsoup4 lxml tqdm openpyxl

import re
import time
import random
import hashlib
import requests
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
from urllib.parse import urljoin

pd.set_option("display.max_columns", 120)


настройки страничек яндекс.недвижимости

In [ ]:
BASE = "https://realty.yandex.ru"

SEGMENTS = {
    "studio": "https://realty.yandex.ru/moskva/snyat/kvartira/studiya/",
    "1": "https://realty.yandex.ru/moskva/snyat/kvartira/odnokomnatnaya/",
    "2": "https://realty.yandex.ru/moskva/snyat/kvartira/dvuhkomnatnaya/",
    "3": "https://realty.yandex.ru/moskva/snyat/kvartira/tryohkomnatnaya/",
    "4plus": "https://realty.yandex.ru/moskva/snyat/kvartira/4-i-bolee/",
}

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0 Safari/537.36"
    ),
    "Accept-Language": "ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7",
}

def fix_mojibake(text):
    # текст в русский
    try:
        return text.encode("latin1").decode("utf-8")
    except Exception:
        return text

def get_html(url):
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    html = r.content.decode("utf-8", errors="replace")
    if "Ð" in html or "Ñ" in html:
        html = fix_mojibake(html)
    time.sleep(random.uniform(1.0, 2.5))
    return html


def clean_num(x):
    if x is None or pd.isna(x):
        return np.nan
    x = str(x).replace("\xa0", " ").replace(",", ".")
    x = re.sub(r"[^\d.]", "", x)
    return float(x) if x else np.nan

def has_words(text, words):
    text = str(text).lower().replace("\xa0", " ")
    return int(any(w.lower() in text for w in words))


для доставания признаков квартир из их описания

In [ ]:
def normalize_text(x):
    x = str(x)
    x = x.replace("\xa0", " ").replace("\u202f", " ")
    x = x.replace("м²", "м2")
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def extract_title_from_block(block):
    patterns = [
        r"([\d,.]+\s*м2\s*·\s*(?:\d+-комнатная|квартира-студия|апартаменты-студия|студия|[\d]+-комнатные апартаменты|[\d]+-комнатная квартира).*?\s*этаж\s*из\s*\d+)",
        r"([\d,.]+\s*м2\s*.*?(?:квартира|апартаменты).*?\s*\d+\s*этаж\s*из\s*\d+)"
    ]

    for p in patterns:
        m = re.search(p, block, flags=re.I)
        if m:
            return normalize_text(m.group(1))

    return np.nan

def parse_title_v2(title):
    t = normalize_text(title).lower().replace(",", ".")

    total_meters = np.nan
    m_area = re.search(r"([\d.]+)\s*м2", t)
    if m_area:
        total_meters = clean_num(m_area.group(1))

    rooms_count = np.nan
    if "студ" in t:
        rooms_count = 0
    else:
        m_rooms = re.search(r"(\d+)\s*[- ]?\s*комнат", t)
        if m_rooms:
            rooms_count = float(m_rooms.group(1))

    floor = np.nan
    floors_count = np.nan
    m_floor = re.search(r"(\d+)\s*этаж\s*из\s*(\d+)", t)
    if m_floor:
        floor = float(m_floor.group(1))
        floors_count = float(m_floor.group(2))

    is_apartment = int("апартамент" in t)

    return rooms_count, total_meters, floor, floors_count, is_apartment

def extract_price_v2(block):
    m = re.search(r"(\d[\d\s]{3,})\s*₽\s*в\s*месяц", block, flags=re.I)
    if m:
        return clean_num(m.group(1))
    return np.nan

def extract_metro_v2(block):
    # чтоб достать метро
    m = re.search(r"([А-Яа-яЁёA-Za-z0-9 \-]+)\s+(\d+)\s*мин", block)
    if m:
        metro = normalize_text(m.group(1))
        metro = metro.split(" ")[-3:]
        metro = " ".join(metro)
        return metro, clean_num(m.group(2))
    return np.nan, np.nan

def extract_repair_type(text):
    text = normalize_text(text).lower()
    if "дизайнерск" in text:
        return "designer"
    if "евроремонт" in text:
        return "euro"
    if "косметическ" in text:
        return "cosmetic"
    if "без ремонт" in text:
        return "no_repair"
    return np.nan

def parse_yandex_page_v2(url, segment_name, page):
    html = get_html(url)
    soup = BeautifulSoup(html, "lxml")
    text = normalize_text(soup.get_text(" ", strip=True))
    # режем страницу на куски по ценам
    price_matches = list(re.finditer(r"\d[\d\s]{3,}\s*₽\s*в\s*месяц", text))

    rows = []

    for k, m in enumerate(price_matches):
        start = max(0, m.start() - 900)
        end = min(len(text), m.end() + 500)
        block = text[start:end]

        price = extract_price_v2(block)
        title = extract_title_from_block(block)

        if pd.isna(title) or pd.isna(price):
            continue

        rooms_count, total_meters, floor, floors_count, is_apartment = parse_title_v2(title)

        if pd.isna(total_meters):
            continue

        underground, metro_time_min = extract_metro_v2(block)

        row = {
            "source": "yandex_realty",
            "segment": segment_name,
            "page": page,
            "url": np.nan,
            "title": title,
            "price": price,
            "rooms_count": rooms_count,
            "total_meters": total_meters,
            "floor": floor,
            "floors_count": floors_count,
            "is_apartment": is_apartment,
            "underground": underground,
            "metro_time_min": metro_time_min,

            # Арендные признаки
            "no_commission": has_words(block, ["без комиссии"]),
            "has_commission": has_words(block, ["комиссия 50", "комиссия 70", "комиссия 80", "комиссия"]),
            "no_deposit": has_words(block, ["без залога"]),
            "has_deposit": has_words(block, ["залог"]),
            "utilities_included": has_words(block, ["цена с ку", "ку включ", "коммунальные платежи включены"]),
            "utilities_extra": has_words(block, ["цена без ку", "ку не включ", "счетчики", "счётчики"]),
            "kids_allowed": has_words(block, ["можно с детьми", "с детьми"]),
            "pets_allowed": has_words(block, ["можно с животными", "с животными", "с питомцами"]),
            "no_pets": has_words(block, ["без животных", "нельзя с животными"]),

            # Оснащение
            "has_furniture": has_words(block, ["мебель", "меблирован", "кухонный гарнитур"]),
            "has_washing_machine": has_words(block, ["стиральная машина"]),
            "has_dryer": has_words(block, ["сушильная машина"]),
            "has_fridge": has_words(block, ["холодильник"]),
            "has_tv": has_words(block, ["телевизор"]),
            "has_conditioner": has_words(block, ["кондиционер"]),
            "has_dishwasher": has_words(block, ["посудомоечная машина"]),
            "has_microwave": has_words(block, ["микроволновка"]),
            "has_boiler": has_words(block, ["бойлер"]),

            "repair_type": extract_repair_type(block),
            "has_3d_tour": has_words(block, ["3d-тур"]),
            "description_text": block[:2500],
        }

        row["offer_id"] = make_id(row)
        rows.append(row)

    return rows


сбор

In [ ]:
all_rows = []

MAX_PAGES_PER_SEGMENT = 10 # 10 страничек => будет +- 1000 квартир

for segment_name, base_url in SEGMENTS.items():
    print(f"\nСобираю сегмент: {segment_name}")

    for page in tqdm(range(1, MAX_PAGES_PER_SEGMENT + 1)):
        url = base_url if page == 1 else f"{base_url}?page={page}"

        try:
            page_rows = parse_yandex_page_v2(url, segment_name, page)
            print(f"segment={segment_name}, page={page}: {len(page_rows)} объявлений")

            all_rows.extend(page_rows)

            pd.DataFrame(all_rows).to_csv(
                "yandex_rent_temp.csv",
                index=False,
                encoding="utf-8-sig"
            )

        except Exception as e:
            print(f"Ошибка segment={segment_name}, page={page}: {e}")
            time.sleep(10)

df_raw = pd.DataFrame(all_rows)

if "offer_id" in df_raw.columns:
    df_raw = df_raw.drop_duplicates(subset=["offer_id"])

print("Размер сырого датасета:", df_raw.shape)

if df_raw.shape[0] > 0:
    display(df_raw[["segment", "title", "price", "rooms_count", "total_meters", "floor", "floors_count"]].head(20))
else:
    print("Пока 0 строк.")



Собираю сегмент: studio


  0%|          | 0/10 [00:00<?, ?it/s]

Ошибка segment=studio, page=1: name 'make_id' is not defined


 10%|█         | 1/10 [00:14<02:12, 14.69s/it]

Ошибка segment=studio, page=2: name 'make_id' is not defined


...

Ошибка segment=4plus, page=9: name 'make_id' is not defined


 90%|█████████ | 9/10 [02:04<00:13, 13.99s/it]

Ошибка segment=4plus, page=10: name 'make_id' is not defined


100%|██████████| 10/10 [02:18<00:00, 13.86s/it]

Размер сырого датасета: (0, 0)
Пока 0 строк.


для проверки ситуации

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Размер датасета:", df_raw.shape)
display(df_raw.head())

audit = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing_%": (df_raw.isna().mean() * 100).round(1),
    "unique": df_raw.nunique(),
    "non_missing": df_raw.notna().sum()
}).sort_values("missing_%", ascending=False)

print("Аудит столбцов:")
display(audit)

num_cols = df_raw.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Числовые переменные:")
display(df_raw[num_cols].describe().T.round(2))

binary_cols = [
    col for col in df_raw.columns
    if set(df_raw[col].dropna().unique()).issubset({0, 1, 0.0, 1.0})
]

if binary_cols:
    print("Бинарные признаки, доля единиц:")
    display(
        df_raw[binary_cols]
        .mean()
        .sort_values(ascending=False)
        .round(3)
        .to_frame("share_1")
    )

for col in ["source", "segment", "repair_type", "underground"]:
    if col in df_raw.columns:
        print(f"\n{col}:")
        display(df_raw[col].value_counts(dropna=False).head(15))

if "offer_id" in df_raw.columns:
    print("Дубли по offer_id:", df_raw.duplicated("offer_id").sum())

if "url" in df_raw.columns:
    print("Дубли по url:", df_raw.duplicated("url").sum())

if {"price", "total_meters"}.issubset(df_raw.columns):
    bad = df_raw[
        (df_raw["price"].isna()) |
        (df_raw["total_meters"].isna()) |
        (df_raw["price"] < 20_000) |
        (df_raw["price"] > 1_500_000) |
        (df_raw["total_meters"] < 10) |
        (df_raw["total_meters"] > 500)
    ]
    print("плохих строк по цене/площади:", bad.shape[0])

for col in ["price", "total_meters", "rooms_count", "floor", "floors_count", "metro_time_min"]:
    if col in df_raw.columns:
        df_raw[col].dropna().hist(bins=40, figsize=(6, 3))
        plt.title(col)
        plt.show()


Размер датасета: (0, 0)


""


Аудит столбцов:


,dtype,missing_%,unique,non_missing


Числовые переменные:


ValueError: Cannot describe a DataFrame without columns

очистка от дублей (вдруг), и квартир где нет информации по главным перменным (цена, площадь...)

In [ ]:
import numpy as np
import pandas as pd

df = df_raw.copy()

# дубли
if "offer_id" in df.columns:
    df = df.drop_duplicates(subset=["offer_id"])

# если без ключевых переменных
df = df.dropna(subset=["price", "total_meters", "rooms_count", "floor", "floors_count"])

# слишком большие/маленькие значения
df = df[
    (df["price"].between(20_000, 1_500_000)) &
    (df["total_meters"].between(10, 500)) &
    (df["rooms_count"].between(0, 7)) &
    (df["floor"] >= 1) &
    (df["floors_count"] >= 1) &
    (df["floor"] <= df["floors_count"]) &
    (df["metro_time_min"].between(0, 60))
].copy()

df["has_deposit"] = 1 - df["no_deposit"]

# создание лог. и производных переменных
df["price_per_meter"] = df["price"] / df["total_meters"]
df["ln_price"] = np.log(df["price"])
df["ln_total_meters"] = np.log(df["total_meters"])

df["is_first_floor"] = (df["floor"] == 1).astype(int)
df["is_last_floor"] = (df["floor"] == df["floors_count"]).astype(int)
df["relative_floor"] = df["floor"] / df["floors_count"]

# заполняем пропуски в ремонте
df["repair_type"] = df["repair_type"].fillna("unknown")

# Оставляем только has_deposit и no_commision (убрали no_deposit и has_comission)
final_cols = [
    "offer_id",
    "segment",
    "title",
    "price",
    "ln_price",
    "price_per_meter",
    "rooms_count",
    "total_meters",
    "ln_total_meters",
    "floor",
    "floors_count",
    "relative_floor",
    "is_first_floor",
    "is_last_floor",
    "metro_time_min",
    "is_apartment",

    # Арендные признаки
    "no_commission",
    "has_deposit",
    "utilities_included",
    "utilities_extra",
    "kids_allowed",
    "pets_allowed",

    # Оснащение квартиры
    "has_furniture",
    "has_washing_machine",
    "has_dryer",
    "has_fridge",
    "has_tv",
    "has_conditioner",
    "has_dishwasher",
    "has_microwave",
    "has_boiler",
    "repair_type",
    "description_text"
]

final_cols = [col for col in final_cols if col in df.columns]

df_final = df[final_cols].copy()

print("Размер после очистки:", df_final.shape)
display(df_final.head())

print("Доли бинарных признаков:")
binary_cols = [
    col for col in df_final.columns
    if set(df_final[col].dropna().unique()).issubset({0, 1, 0.0, 1.0})
]
display(df_final[binary_cols].mean().sort_values(ascending=False).round(3))


сохраняем с таблицу

In [ ]:
df_final.to_csv(
    "moscow_rent_yandex_final_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

df_final.to_excel(
    "moscow_rent_yandex_final_clean.xlsx",
    index=False
)

## редактирование переменных

нужна переменная для расстояния до центра, так как оказалось, что координаты квартир прямо не указаны в объявлениях. высчитываем значение новой переменной center_distance как сумму расстояния до метро (5км/ч * время до метро) и расстояния от этого метро до центра (координаты станций берем из википедии, координаты центра взяли 55.7558; 37.6173)

In [ ]:
import pandas as pd
import requests
import re
import math
from urllib.parse import quote

df = pd.read_excel("moscow_rent.xlsx")

url = "https://ru.wikipedia.org/wiki/Список_станций_Московского_метрополитена"

headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
response.encoding = "utf-8"

tables = pd.read_html(response.text)

metro_tables = []
for t in tables:
    cols = " ".join(map(str, t.columns))
    if "Название" in cols and "Координаты" in cols:
        metro_tables.append(t)

df_metro = metro_tables[0].copy()
df_metro.columns = [str(c).replace("\n", " ").strip() for c in df_metro.columns]

name_col = [c for c in df_metro.columns if "Название" in c][0]
coord_col = [c for c in df_metro.columns if "Координаты" in c][0]

def clean_station_name(x):
    x = str(x)
    x = re.sub(r"\[.*?\]", "", x)
    x = re.sub(r"«|»", "", x)
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def dms_to_decimal(x):
    if pd.isna(x):
        return None

    x = str(x)
    nums = re.findall(r"\d+", x)

    if len(nums) < 6:
        return None

    lat = int(nums[0]) + int(nums[1]) / 60 + int(nums[2]) / 3600
    lon = int(nums[3]) + int(nums[4]) / 60 + int(nums[5]) / 3600

    return lat, lon

df_metro["metro_station"] = df_metro[name_col].apply(clean_station_name)
df_metro["coords"] = df_metro[coord_col].apply(dms_to_decimal)

df_metro = df_metro.dropna(subset=["coords"]).drop_duplicates("metro_station")

metro_coords = df_metro.set_index("metro_station")["coords"].to_dict()

def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x).lower()
    x = x.replace("ё", "е")
    x = re.sub(r"[^а-яa-z0-9\s-]", " ", x)
    x = re.sub(r"\s+", " ", x)
    return x.strip()

metro_coords_norm = {
    normalize_text(station): coords
    for station, coords in metro_coords.items()
}

station_names = sorted(metro_coords_norm.keys(), key=len, reverse=True)

def extract_metro_station(text):
    text_norm = normalize_text(text)

    for station in station_names:
        pattern = r"(^|\s)" + re.escape(station) + r"($|\s)"
        if re.search(pattern, text_norm):
            return station

    return None

def haversine(lat1, lon1, lat2, lon2):
    r = 6371

    lat1, lon1, lat2, lon2 = map(
        math.radians,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    )

    return 2 * r * math.asin(math.sqrt(a))

center_lat = 55.7558
center_lon = 37.6173

df["metro_station_text"] = df["description_text"].apply(extract_metro_station)

df["metro_lat"] = df["metro_station_text"].map(
    lambda x: metro_coords_norm[x][0] if pd.notna(x) and x in metro_coords_norm else None
)

df["metro_lon"] = df["metro_station_text"].map(
    lambda x: metro_coords_norm[x][1] if pd.notna(x) and x in metro_coords_norm else None
)

df["center_distance"] = df.apply(
    lambda row: haversine(
        row["metro_lat"],
        row["metro_lon"],
        center_lat,
        center_lon
    )
    if pd.notna(row["metro_lat"]) and pd.notna(row["metro_lon"])
    else None,
    axis=1
)

df["distance_to_metro"] = 5 * df["metro_time_min"] / 60

df["center_distance_with_walk"] = df["center_distance"] + df["distance_to_metro"]

df[[
    "description_text",
    "metro_station_text",
    "metro_lat",
    "metro_lon",
    "center_distance",
    "distance_to_metro",
    "center_distance_with_walk"
]].head()

/tmp/ipykernel_4919/1251074487.py:16: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


,description_text,metro_station_text,metro_lat,metro_lon,center_distance,distance_to_metro,center_distance_with_walk
0,ду в Москве ? На Яндекс Недвижимости в Москве ...,римская,55.746389,37.681944,4.178563,0.666667,4.845229
1,рг без комиссии Снять Написать в чат 3 часа на...,бунинская аллея,55.538056,37.515833,25.035148,1.583333,26.618482
2,"одольск , агентство 35 000 ₽ в месяц онлайн по...",бунинская аллея,55.538056,37.515833,25.035148,0.833333,25.868482
3,"вка Дом - монолитный, окна выходят во двор. Ес...",бунинская аллея,55.538056,37.515833,25.035148,1.000000,26.035148
4,"олодильник Микроволновка Дом - монолитный, окн...",бутырская,55.813333,37.602778,6.461532,0.583333,7.044866


In [ ]:
df_out = pd.read_excel("moscow_rent.xlsx")
df_out["center_distance"] = df["center_distance"]
df_out.to_excel("moscow_rent.xlsx", index=False)

repair_type -> дамми переменная (unknown - базовая категория)

In [ ]:
df["repair_designer"] = (df["repair_type"] == "designer").astype(int)
df["repair_euro"] = (df["repair_type"] == "euro").astype(int)

df.to_excel("moscow_rent.xlsx", index=False)

за кадром в экселе сделали удаление лишних столбцов (текстовое описание объявления, например)

чистка строк с пропусками, и сохраняем итоговый датасет

In [ ]:
df = pd.read_excel("moscow_rent (3).xlsx")
df = df.dropna()
df.to_excel("data.xlsx", index=False)